# ClarioAI System Test Notebook

Notebook ini menguji seluruh komponen sistem ClarioAI secara terstruktur:

| # | Seksi | Deskripsi |
|---|-------|-----------|
| 1 | Imports & Environment | Verifikasi semua package tersedia |
| 2 | State Schema | Dataclass dan EBPState |
| 3 | Utility Functions | `extract_json`, `format_constraints` |
| 4 | LLM Connectivity | Koneksi ke DeepInfra / Qwen |
| 5 | Internet Search Tool | BrightData API |
| 6 | MongoDB | Koneksi, save, load state |
| 7 | Agent Nodes | Market Scout, Strategic Architect, Financial Analyst, Ethics Agent, Orchestrator |
| 8 | Graph Pipeline | Build graph & jalankan 1 iterasi penuh |
| 9 | End-to-End | Full run dengan business scenario nyata |

---
## 1. Imports & Environment Check

In [1]:
import sys, importlib

REQUIRED_PACKAGES = [
    "langchain",
    "langchain_openai",
    "langchain_core",
    "langgraph",
    "pymongo",
    "requests",
]

missing = []
for pkg in REQUIRED_PACKAGES:
    try:
        importlib.import_module(pkg)
        print(f"  [OK] {pkg}")
    except ImportError:
        print(f"  [MISSING] {pkg}")
        missing.append(pkg)

if missing:
    print(f"\nInstall missing: pip install {' '.join(missing)}")
else:
    print("\nSemua package tersedia.")

  [OK] langchain
  [OK] langchain_openai
  [OK] langchain_core
  [OK] langgraph
  [OK] pymongo
  [OK] requests

Semua package tersedia.


In [2]:
# Pastikan root project ada di sys.path agar import relatif berjalan
import os
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")

Project root: d:\Gabriel\Gabriel\Telkom University\Kuliah\Semester VI\Capstone\Multi-Agent-Decision-Support-System


---
## 2. State Schema Tests

In [3]:
from states.schema import (
    BussinessConstraints,
    MarketScoutReport,
    StrategicReport,
    FinancialAnalysisReport,
    EthicsAnalysisReport,
    EBPState,
)
print("Import states.schema: OK")

Import states.schema: OK


In [4]:
# --- Test BussinessConstraints (intentional typo sesuai codebase) ---
constraints = BussinessConstraints(
    sector_and_domain="EdTech / Platform Belajar Online",
    audience="Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)",
    initial_prompt="Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional",
)
print("BussinessConstraints:")
print(f"  sector_and_domain : {constraints.sector_and_domain}")
print(f"  audience          : {constraints.audience}")
print(f"  initial_prompt    : {constraints.initial_prompt}")

BussinessConstraints:
  sector_and_domain : EdTech / Platform Belajar Online
  audience          : Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)
  initial_prompt    : Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional


In [ ]:
# --- Test report dataclasses ---
market_report = MarketScoutReport(
    ideas=["Ide A", "Ide B"],
    agent_explanation="Penjelasan dari market scout.",
    sources=["internal_brainstorm", "external_search"],
)
strategic_report = StrategicReport(
    swot_analysis="SWOT placeholder",
    pastel_analysis="PESTEL placeholder",
)
financial_report = FinancialAnalysisReport(analysis_result="Proyeksi keuangan placeholder")
ethics_report    = EthicsAnalysisReport(analysis_result="Analisis etika placeholder")

print("MarketScoutReport  :", market_report)
print("StrategicReport    :", strategic_report)
print("FinancialAnalysis  :", financial_report)
print("EthicsAnalysis     :", ethics_report)

TypeError: MarketScoutReport.__init__() missing 1 required positional argument: 'sources'

In [6]:
# --- Test EBPState instantiation ---
import uuid

sample_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "user_id": "test_user_notebook",
    "bussiness_constraints": constraints,
    "market_scout_report": None,
    "strategic_report": None,
    "financial_analysis_report": None,
    "ethics_analysis_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "messages": [],
    "iteration": 0,
    "max_iterations": 3,
    "user_feedback": None,
}
print("EBPState state_id :", sample_state["state_id"])
print("approval_status   :", sample_state["approval_status"])
print("iteration         :", sample_state["iteration"], "/", sample_state["max_iterations"])
print("market_report None:", sample_state["market_scout_report"] is None)

EBPState state_id : 76ffa243-ec1b-4201-a260-9f1de3a39217
approval_status   : pending
iteration         : 0 / 3
market_report None: True


---
## 3. Utility Functions Tests

In [7]:
from functions.agent_utils import extract_json, format_constraints
print("Import functions.agent_utils: OK")

Import functions.agent_utils: OK


In [8]:
# --- extract_json: markdown fenced ---
fenced = '''
Berikut hasilnya:
```json
{"status": "approved", "score": 8.5, "reason": "Market viable"}
```
'''
result = extract_json(fenced)
assert result["status"] == "approved", "Fenced JSON parse gagal"
assert result["score"] == 8.5
print("extract_json (fenced)  :", result)

extract_json (fenced)  : {'status': 'approved', 'score': 8.5, 'reason': 'Market viable'}


In [9]:
# --- extract_json: bare JSON ---
bare = 'Analisis selesai. {"ideas": ["A", "B"], "confidence": 0.9}'
result2 = extract_json(bare)
assert result2["ideas"] == ["A", "B"]
print("extract_json (bare)    :", result2)

extract_json (bare)    : {'ideas': ['A', 'B'], 'confidence': 0.9}


In [10]:
# --- extract_json: fallback ke {"raw": ...} ---
plain = "Tidak ada JSON di sini sama sekali."
result3 = extract_json(plain)
assert "raw" in result3
print("extract_json (fallback):", result3)

extract_json (fallback): {'raw': 'Tidak ada JSON di sini sama sekali.'}


In [11]:
# --- format_constraints ---
formatted = format_constraints(constraints)
print("format_constraints output:")
print(formatted)
assert "EdTech" in formatted
assert "Pelajar SMA" in formatted

format_constraints output:
Sector/Domain: EdTech / Platform Belajar Online
Target Audience: Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)
Business Idea: Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional


In [12]:
# --- format_constraints dengan None ---
none_fmt = format_constraints(None)
assert "No constraints" in none_fmt
print("format_constraints(None):", none_fmt)

format_constraints(None): No constraints provided.


---
## 4. LLM Connectivity Test

> Sel ini melakukan satu panggilan ke DeepInfra API. Perlu koneksi internet.

In [13]:
from functions.llm import get_llm, MODEL_NAME, DEEPINFRA_BASE_URL
print(f"Model      : {MODEL_NAME}")
print(f"Base URL   : {DEEPINFRA_BASE_URL}")

Model      : Qwen/Qwen3.5-397B-A17B
Base URL   : https://api.deepinfra.com/v1/openai


In [14]:
llm = get_llm(temperature=0.3)

from langchain_core.messages import HumanMessage

response = llm.invoke([HumanMessage(content="Balas hanya dengan kata: PING")])
print("LLM response:", response.content[:200])

LLM response: 

PING


---
## 5. Internet Search Tool Test

> Memerlukan koneksi internet dan BrightData API key yang valid.

In [15]:
from tools.internet_search import internet_search
print("Import tools.internet_search: OK")
print("Tool name :", internet_search.name)
print("Tool desc :", internet_search.description[:80], "...")

Import tools.internet_search: OK
Tool name : internet_search
Tool desc : Search the internet for real-time business and market information.
    Use this  ...


In [16]:
query = "Ide bisnis kuliner"
print(f"Searching: '{query}' ...\n")

search_result = internet_search.invoke({"query": query})
# print(search_result[:1000])  # Tampilkan 1000 char pertama

[03:04:45] [internet_search] START query='Ide bisnis kuliner'
[03:04:45] Starting new HTTPS connection (1): api.brightdata.com:443


Searching: 'Ide bisnis kuliner' ...



[03:04:47] https://api.brightdata.com:443 "POST /request HTTP/1.1" 200 None
[03:04:47] [internet_search] BrightData search took 2.80s, status=200
[03:04:47] [internet_search] Parsed 5 results
[03:04:47] Starting new HTTPS connection (1): ocean.bca.co.id:443
[03:04:47] Starting new HTTPS connection (1): www.unileverfoodsolutions.co.id:443
[03:04:47] Starting new HTTPS connection (1): www.foliopos.com:443
[03:04:47] Starting new HTTPS connection (1): www.lalamove.com:443
[03:04:47] Starting new HTTPS connection (1): www.cimbniaga.co.id:443
[03:04:49] https://ocean.bca.co.id:443 "GET /artikel/10-ide-bisnis-kuliner HTTP/1.1" 307 None
[03:04:49] https://www.lalamove.com:443 "GET /id/blog/bisnis-kuliner-modal-kecil/ HTTP/1.1" 200 None
[03:04:49] [_fetch_page_text] HTTP GET took 1.39s, status=200, url=https://www.lalamove.com/id/blog/bisnis-kuliner-modal-kecil/
[03:04:49] https://ocean.bca.co.id:443 "GET /id/artikel/10-ide-bisnis-kuliner HTTP/1.1" 200 None
[03:04:49] [_fetch_page_text] HTTP G

In [17]:
print(search_result)

Search results for 'Ide bisnis kuliner':


1. 30 Ide Jualan Makanan Kekinian Untuk Bisnis Kuliner!
   https://www.unileverfoodsolutions.co.id/id/inspirasi-chef/ukm-sukses/10-ide-jualan-makanan-kekinian-untuk-bisnis-kuliner.html

2. 10 Ide Bisnis Kuliner Kekinian, Bisa Dicoba untuk Pemula
   https://ocean.bca.co.id/artikel/10-ide-bisnis-kuliner

3. 23 Ide Bisnis Kuliner Modal Kecil Tapi Cuan Besar!
   https://www.lalamove.com/id/blog/bisnis-kuliner-modal-kecil/

4. 25 Ide Usaha Kuliner Rumahan yang Mudah dan ...
   https://www.foliopos.com/blog/detail/25-ide-usaha-kuliner-rumahan-yang-mudah-dan-menguntungkan

5. Ide Usaha Makanan Kekinian dan Strategi Bisnisnya
   https://www.cimbniaga.co.id/id/inspirasi/bisnis/ide-usaha-makanan-kekinian-dan-strategi-bisnisnya


-- Page Summaries --

[https://ocean.bca.co.id/artikel/10-ide-bisnis-kuliner]
Here’s a concise summary of the webpage content relevant to the search query “Ide bisnis kuliner”:

**The webpage provides 10 ideas for starting a suc

In [18]:
stop

NameError: name 'stop' is not defined

---
## 6. MongoDB Tests

> Memerlukan MongoDB berjalan di `localhost:27017`.

In [19]:
from functions.mongodb import create_new_state, save_state, load_state, MONGO_URI, DB_NAME
print(f"MongoDB URI : {MONGO_URI}")
print(f"Database    : {DB_NAME}")

MongoDB URI : mongodb://localhost:27017
Database    : clario_ai


In [21]:
# --- Test koneksi MongoDB ---
from pymongo import MongoClient
try:
    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=3000)
    client.admin.command("ping")
    print("MongoDB connection: OK")
    MONGO_AVAILABLE = True
except Exception as e:
    print(f"MongoDB connection: GAGAL ({e})")
    print("Test seksi ini di-skip. Jalankan MongoDB terlebih dahulu.")
    MONGO_AVAILABLE = False

MongoDB connection: OK


In [22]:
if MONGO_AVAILABLE:
    # --- create_new_state ---
    new_state = create_new_state(
        constraints=constraints,
        user_id="notebook_test_user",
        max_iterations=3,
    )
    print("State dibuat, ID:", new_state["state_id"])
    print("approval_status :", new_state["approval_status"])
    print("iteration       :", new_state["iteration"])
    TEST_STATE_ID = new_state["state_id"]
else:
    TEST_STATE_ID = None
    print("Skip — MongoDB tidak tersedia.")

State dibuat, ID: c81ad92d-1e19-4504-9861-76544cd2aa59
approval_status : pending
iteration       : 0


In [23]:
if MONGO_AVAILABLE:
    # --- save_state ---
    new_state["market_scout_report"] = market_report
    saved_id = save_state(new_state)
    print("save_state OK, returned ID:", saved_id)
else:
    print("Skip.")

save_state OK, returned ID: c81ad92d-1e19-4504-9861-76544cd2aa59


In [24]:
if MONGO_AVAILABLE:
    # --- load_state ---
    loaded = load_state(TEST_STATE_ID)
    assert loaded is not None, "State tidak ditemukan setelah save!"
    assert loaded["state_id"] == TEST_STATE_ID
    assert loaded["market_scout_report"] is not None
    assert loaded["market_scout_report"].ideas == ["Ide A", "Ide B"]
    print("load_state OK")
    print("  state_id        :", loaded["state_id"])
    print("  user_id         :", loaded["user_id"])
    print("  market ideas    :", loaded["market_scout_report"].ideas)

    # --- load non-existent ---
    not_found = load_state("00000000-0000-0000-0000-000000000000")
    assert not_found is None
    print("  load non-existent: None (benar)")
else:
    print("Skip.")

load_state OK
  state_id        : c81ad92d-1e19-4504-9861-76544cd2aa59
  user_id         : notebook_test_user
  market ideas    : ['Ide A', 'Ide B']
  load non-existent: None (benar)


In [25]:
if MONGO_AVAILABLE:
    # --- Serialisasi messages (HumanMessage, AIMessage, SystemMessage) ---
    from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
    from functions.mongodb import _serialize_messages, _deserialize_messages

    msgs = [
        SystemMessage(content="System prompt"),
        HumanMessage(content="Halo"),
        AIMessage(content="Halo juga!"),
    ]
    serialized = _serialize_messages(msgs)
    deserialized = _deserialize_messages(serialized)

    assert len(deserialized) == 3
    assert deserialized[1].content == "Halo"
    print("Message serialization/deserialization: OK")
    print("  Types:", [type(m).__name__ for m in deserialized])
else:
    print("Skip.")

Message serialization/deserialization: OK
  Types: ['SystemMessage', 'HumanMessage', 'AIMessage']


---
## 7. Agent Node Tests

Setiap agent diuji secara **terpisah** dengan state minimal. Ini menguji logika prompt + parsing output tiap node tanpa menjalankan graph penuh.

> Sel-sel ini memanggil LLM secara nyata — estimasi 30-120 detik per agent.

In [26]:
# Helper: buat state bersih untuk setiap test
def make_fresh_state(iteration=0, with_market=False, with_strategic=False, with_financial=False, with_ethics=False):
    state: EBPState = {
        "state_id": str(uuid.uuid4()),
        "user_id": "notebook_agent_test",
        "bussiness_constraints": constraints,
        "market_scout_report": market_report if with_market else None,
        "strategic_report": strategic_report if with_strategic else None,
        "financial_analysis_report": financial_report if with_financial else None,
        "ethics_analysis_report": ethics_report if with_ethics else None,
        "approval_status": "pending",
        "orchestrator_feedback": None,
        "messages": [],
        "iteration": iteration,
        "max_iterations": 3,
        "user_feedback": None,
    }
    return state

print("Helper make_fresh_state: siap")

Helper make_fresh_state: siap


### 7.1 Market Scout Agent

In [27]:
from nodes.market_scout import market_scout_node

state_ms = make_fresh_state(iteration=0)
print("Menjalankan Market Scout Agent...")
result_ms = market_scout_node(state_ms)

report_ms = result_ms.get("market_scout_report")
assert report_ms is not None, "market_scout_report seharusnya tidak None!"
assert isinstance(report_ms.ideas, list), "ideas harus berupa list"
assert len(report_ms.ideas) > 0, "ideas tidak boleh kosong"

print("\n[Market Scout] LULUS")
print(f"  Jumlah ide       : {len(report_ms.ideas)}")
for i, idea in enumerate(report_ms.ideas, 1):
    print(f"  Ide {i}: {idea[:100]}")
print(f"\n  Penjelasan (150 char): {report_ms.agent_explanation[:150]}...")

2026-05-27 03:06:32.960  DEBUG  [clario.market_scout]  ============================================================
2026-05-27 03:06:32.962  DEBUG  [clario.market_scout]  → Market Scout Agent dimulai
2026-05-27 03:06:32.991  DEBUG  [clario.market_scout]  [PLAN] Merencanakan 5 query pencarian...
[03:06:33] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-18984ad5-321b-4062-b8a6-03fda4d9bca0', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Market Scout Agent in a multi-agent AI business planning system.\nYour mission is to perform real-time market research and identify viable business opportunities\nfor the given business constraints.\n\nSearch results will be provided for you in bulk. Once you have them, synthesise all findings\ninto a comprehensive MarketScoutReport. If critical data is still missing after rev

Menjalankan Market Scout Agent...


[03:06:33] connect_tcp.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000002C9C7A93470>
[03:06:33] start_tls.started ssl_context=<ssl.SSLContext object at 0x000002C9C5FCEE50> server_hostname='api.deepinfra.com' timeout=None
[03:06:33] start_tls.complete return_value=<httpcore._backends.sync.SyncStream object at 0x000002C9C64402F0>
[03:06:33] send_request_headers.started request=<Request [b'POST']>
[03:06:33] send_request_headers.complete
[03:06:33] send_request_body.started request=<Request [b'POST']>
[03:06:33] send_request_body.complete
[03:06:33] receive_response_headers.started request=<Request [b'POST']>
[03:07:42] receive_response_headers.complete return_value=(b'HTTP/1.1', 200, b'OK', [(b'Date', b'Tue, 26 May 2026 20:07:41 GMT'), (b'Content-Type', b'application/json'), (b'Content-Length', b'14580'), (b'Connection', b'keep-alive'), (b'server', b'uvicorn'), (b'x-robots-tag', b'noindex'), (b'x-request-id', b'RyWZYzm3KXKY6hikgPxxRKCU')])
[03:07:42] HTTP Reques


[Market Scout] LULUS
  Jumlah ide       : 3
  Ide 1: Platform Micro-Learning SNBT Berbasis Video Pendek (TikTok-style) dengan Fitur Gamifikasi dan AI Per
  Ide 2: Platform Persiapan Sertifikasi Profesional Mikro (CFA, ACCA, TOEFL/IELTS) untuk Mahasiswa Tingkat Ak
  Ide 3: Layanan B2B2C 'Micro-Course' untuk Sekolah dan Universitas (Kurikulum Merdeka): Menyediakan konten v

  Penjelasan (150 char): Berdasarkan riset pasar real-time, lanskap EdTech Indonesia untuk segmen persiapan ujian (test preparation) sangat menjanjikan dengan proyeksi nilai p...


### 7.2 Strategic Architect Agent

In [ ]:
from nodes.strategic_architect import strategic_architect_node

# Strategic Architect butuh market_scout_report
state_sa = make_fresh_state(iteration=0, with_market=True)
# Gunakan hasil nyata dari market scout jika tersedia
if report_ms is not None:
    state_sa["market_scout_report"] = report_ms

print("Menjalankan Strategic Architect Agent...")
result_sa = strategic_architect_node(state_sa)

report_sa = result_sa.get("strategic_report")
assert report_sa is not None, "strategic_report seharusnya tidak None!"
assert report_sa.swot_analysis, "swot_analysis tidak boleh kosong"
assert report_sa.pastel_analysis, "pastel_analysis tidak boleh kosong"

print("\n[Strategic Architect] LULUS")
print(f"  SWOT (200 char)  : {report_sa.swot_analysis[:200]}...")
print(f"  PESTEL (200 char): {report_sa.pastel_analysis[:200]}...")

2026-05-27 03:08:51.409  DEBUG  [clario.strategic_architect]  ============================================================
2026-05-27 03:08:51.410  DEBUG  [clario.strategic_architect]  → Strategic Architect Agent dimulai
2026-05-27 03:08:51.419  DEBUG  [clario.strategic_architect]  [PLAN] Merencanakan 5 query pencarian...
[03:08:51] Request options: {'method': 'post', 'url': '/chat/completions', 'headers': {'X-Stainless-Raw-Response': 'true'}, 'files': None, 'idempotency_key': 'stainless-python-retry-ca815d80-eb6c-430e-abd7-69b8a8a6b675', 'security': {'bearer_auth': True}, 'content': None, 'json_data': {'messages': [{'content': 'You are the Strategic Architect Agent in a multi-agent AI business planning system.\nYou create data-backed strategic analyses for new business ventures.\n\nYour tasks:\n1. Perform a thorough SWOT Analysis (Strengths, Weaknesses, Opportunities, Threats)\n2. Perform a thorough PESTEL Analysis (Political, Economic, Social, Technological, Environmental, Legal)\n\n

Menjalankan Strategic Architect Agent...


### 7.3 Financial Analyst Agent

In [ ]:
from nodes.financial_analyst import financial_analyst_node

state_fa = make_fresh_state(iteration=0, with_market=True, with_strategic=True)
# state_fa['market_scout_report'] = None  
state_fa["market_scout_report"] = report_ms
if report_sa is not None:
    state_fa["strategic_report"] = report_sa

print("Menjalankan Financial Analyst Agent...")
result_fa = financial_analyst_node(state_fa)

report_fa = result_fa.get("financial_analysis_report")
assert report_fa is not None, "financial_analysis_report seharusnya tidak None!"
assert report_fa.analysis_result, "analysis_result tidak boleh kosong"

print("\n[Financial Analyst] LULUS")
print(f"  Hasil (300 char) : {report_fa.analysis_result[:300]}...")

2026-05-14 13:09:23.924  DEBUG  [clario.financial_analyst]  ============================================================
2026-05-14 13:09:23.924  DEBUG  [clario.financial_analyst]  → Financial Analyst Agent dimulai
2026-05-14 13:09:23.931  DEBUG  [clario.financial_analyst]  [LLM] Round 1 — memanggil LLM...


Menjalankan Financial Analyst Agent...


2026-05-14 13:09:34.518  DEBUG  [clario.financial_analyst]  [LLM] Round 1 selesai dalam 10.59s | 5 tool call(s)
2026-05-14 13:09:34.519  DEBUG  [clario.financial_analyst]  [TOOL] → internet_search(query="Indonesia EdTech startup salaries operational costs 2024 2025 developer teacher ...")
2026-05-14 13:11:10.414  DEBUG  [clario.financial_analyst]  [TOOL] ✓ internet_search selesai dalam 95.89s
2026-05-14 13:11:10.415  DEBUG  [clario.financial_analyst]  [TOOL] → internet_search(query="Indonesia EdTech pricing subscription model Ruangguru Zenius Brainly harga berla...")
2026-05-14 13:11:45.849  DEBUG  [clario.financial_analyst]  [TOOL] ✓ internet_search selesai dalam 35.43s
2026-05-14 13:11:45.851  DEBUG  [clario.financial_analyst]  [TOOL] → internet_search(query="Indonesia EdTech customer acquisition cost CAC conversion rate marketing digital...")
2026-05-14 13:13:45.321  DEBUG  [clario.financial_analyst]  [TOOL] ✓ internet_search selesai dalam 119.47s
2026-05-14 13:13:45.322  DEBUG  [cl


[Financial Analyst] LULUS
  Hasil (300 char) : # Financial Analysis: Micro-Learning EdTech Platform Indonesia
## Platform Belajar Online untuk Persiapan SNBT & Sertifikasi Profesional

---

## 1. INITIAL INVESTMENT ESTIMATE

### Startup Costs Breakdown (Year 0)

| Category | Item | Cost (IDR) | Cost (USD)* |
|----------|------|------------|-----...


In [ ]:
print(f"  Hasil (300 char) : {report_fa.analysis_result}...")

  Hasil (300 char) : # Financial Analysis: Micro-Learning EdTech Platform Indonesia
## Platform Belajar Online untuk Persiapan SNBT & Sertifikasi Profesional

---

## 1. INITIAL INVESTMENT ESTIMATE

### Startup Costs Breakdown (Year 0)

| Category | Item | Cost (IDR) | Cost (USD)* |
|----------|------|------------|-------------|
| **Technology Development** | Platform development (mobile app + web) | 850,000,000 | $54,500 |
| | Cloud infrastructure setup (AWS/GCP) | 120,000,000 | $7,700 |
| | Video hosting & CDN integration | 80,000,000 | $5,100 |
| | AI/ML for spaced repetition algorithm | 200,000,000 | $12,800 |
| **Content Production** | Initial video content library (500 videos @ 30-60 sec) | 350,000,000 | $22,400 |
| | SNBT question bank development | 150,000,000 | $9,600 |
| | Professional certification modules (5 tracks) | 180,000,000 | $11,500 |
| **Team Salaries (6 months pre-launch)** | Tech team (3 developers @ 15-25jt/month) | 360,000,000 | $23,000 |
| | Content creators (5

In [ ]:
report_ms.ideas

['Platform micro-learning video pendek (30-60 detik) khusus persiapan SNBT dengan fitur gamifikasi dan spaced repetition - didukung data bahwa 70% penetrasi internet mobile Indonesia dan Gen Z lebih menyukai konten bite-sized yang meningkatkan retensi hingga 40%',
 'Platform sertifikasi profesional mikro untuk skill digital (digital marketing, data analytics, cloud computing) dengan kemitraan industri - memanfaatkan pasar TIC Indonesia USD 561 juta dan 3 juta sertifikasi yang telah diterbitkan pemerintah dengan standar SKKNI',
 'Hybrid learning community dengan mentorship peer-to-peer untuk pelajar SMA-mahasiswa, menggabungkan video micro-learning dengan sesi live Q&A dan tryout berkala - menjawab kebutuhan akan pendekatan terstruktur dan engagement rendah di platform existing']

In [ ]:
print(report_fa.analysis_result)

# Financial Analysis: Micro-Learning EdTech Platform Indonesia
## Platform Belajar Online untuk Persiapan SNBT & Sertifikasi Profesional

---

## 1. INITIAL INVESTMENT ESTIMATE

### Startup Costs Breakdown (Year 0)

| Category | Item | Cost (IDR) | Cost (USD)* |
|----------|------|------------|-------------|
| **Technology Development** | Platform development (mobile app + web) | 850,000,000 | $54,500 |
| | Cloud infrastructure setup (AWS/GCP) | 120,000,000 | $7,700 |
| | Video hosting & CDN integration | 80,000,000 | $5,100 |
| | AI/ML for spaced repetition algorithm | 200,000,000 | $12,800 |
| **Content Production** | Initial video content library (500 videos @ 30-60 sec) | 350,000,000 | $22,400 |
| | SNBT question bank development | 150,000,000 | $9,600 |
| | Professional certification modules (5 tracks) | 180,000,000 | $11,500 |
| **Team Salaries (6 months pre-launch)** | Tech team (3 developers @ 15-25jt/month) | 360,000,000 | $23,000 |
| | Content creators (5 @ 8-12jt/month) | 30

In [ ]:
print(report_fa.analysis_result)

# Financial Analysis: Micro-Learning EdTech Platform Indonesia
## Platform Belajar Online untuk Persiapan SNBT & Sertifikasi Profesional

---

## 1. INITIAL INVESTMENT ESTIMATE

### Startup Costs Breakdown (Year 0)

| Category | Item | Cost (IDR) | Cost (USD)* |
|----------|------|------------|-------------|
| **Technology Development** | Platform development (mobile app + web) | 850,000,000 | $54,500 |
| | Cloud infrastructure setup (AWS/GCP) | 120,000,000 | $7,700 |
| | Video hosting & CDN integration | 80,000,000 | $5,100 |
| | AI/ML for spaced repetition algorithm | 200,000,000 | $12,800 |
| **Content Production** | Initial video content library (500 videos @ 30-60 sec) | 350,000,000 | $22,400 |
| | SNBT question bank development | 150,000,000 | $9,600 |
| | Professional certification modules (5 tracks) | 180,000,000 | $11,500 |
| **Team Salaries (6 months pre-launch)** | Tech team (3 developers @ 15-25jt/month) | 360,000,000 | $23,000 |
| | Content creators (5 @ 8-12jt/month) | 30

### 7.4 Ethics Guardian Agent

In [ ]:
from nodes.ethics_agent import ethics_agent_node

state_ea = make_fresh_state(iteration=0, with_market=True, with_strategic=True, with_financial=True)
state_ea["market_scout_report"] = report_ms
if report_sa: state_ea["strategic_report"] = report_sa
if report_fa: state_ea["financial_analysis_report"] = report_fa

print("Menjalankan Ethics Guardian Agent...")
result_ea = ethics_agent_node(state_ea)

report_ea = result_ea.get("ethics_analysis_report")
assert report_ea is not None, "ethics_analysis_report seharusnya tidak None!"
assert report_ea.analysis_result, "analysis_result tidak boleh kosong"

print("\n[Ethics Guardian] LULUS")
print(f"  Hasil (300 char) : {report_ea.analysis_result[:300]}...")

2026-05-14 13:18:53.828  DEBUG  [clario.ethics_agent]  ============================================================
2026-05-14 13:18:53.829  DEBUG  [clario.ethics_agent]  → Ethics Guardian Agent dimulai


2026-05-14 13:18:53.835  DEBUG  [clario.ethics_agent]  [LLM] Round 1 — memanggil LLM...


Menjalankan Ethics Guardian Agent...


2026-05-14 13:19:05.101  DEBUG  [clario.ethics_agent]  [LLM] Round 1 selesai dalam 11.26s | 5 tool call(s)
2026-05-14 13:19:05.102  DEBUG  [clario.ethics_agent]  [TOOL] → internet_search(query="Indonesia EdTech regulations 2024 2025 licensing requirements NIB OSS Kominfo")
2026-05-14 13:20:19.951  DEBUG  [clario.ethics_agent]  [TOOL] ✓ internet_search selesai dalam 74.85s
2026-05-14 13:20:19.952  DEBUG  [clario.ethics_agent]  [TOOL] → internet_search(query="UU PDP Indonesia 2024 data privacy protection personal data law compliance requi...")


KeyboardInterrupt: 

### 7.5 Lead Orchestrator — iterasi pertama (belum ada laporan)

In [ ]:
from nodes.lead_orchestrator import lead_orchestrator_node

# Iterasi 0: semua report None → orchestrator harus route ke market scout
state_orch_init = make_fresh_state(iteration=0)
print("Menjalankan Lead Orchestrator (iterasi awal, tanpa laporan)...")
result_orch_init = lead_orchestrator_node(state_orch_init)

print("\n[Orchestrator iterasi awal] Output:")
print(f"  approval_status      : {result_orch_init.get('approval_status')}")
print(f"  orchestrator_feedback: {str(result_orch_init.get('orchestrator_feedback', ''))[:150]}")
print(f"  iteration            : {result_orch_init.get('iteration')}")

Menjalankan Lead Orchestrator (iterasi awal, tanpa laporan)...

[Orchestrator iterasi awal] Output:
  approval_status      : pending
  orchestrator_feedback: None
  iteration            : None


In [ ]:
# Iterasi 1: semua report sudah ada → orchestrator evaluasi dan beri verdict
state_orch_eval = make_fresh_state(iteration=1, with_market=True, with_strategic=True, with_financial=True, with_ethics=True)
state_orch_eval["market_scout_report"] = report_ms
if report_sa: state_orch_eval["strategic_report"] = report_sa
if report_fa: state_orch_eval["financial_analysis_report"] = report_fa
if report_ea: state_orch_eval["ethics_analysis_report"] = report_ea

print("Menjalankan Lead Orchestrator (evaluasi laporan lengkap)...")
result_orch_eval = lead_orchestrator_node(state_orch_eval)

status = result_orch_eval.get("approval_status")
assert status in ("approved", "rejected", "pending"), f"Status tidak valid: {status}"

print("\n[Orchestrator evaluasi] LULUS")
print(f"  approval_status      : {status}")
print(f"  iteration            : {result_orch_eval.get('iteration')}")
feedback = result_orch_eval.get('orchestrator_feedback') or ""
print(f"  feedback (200 char)  : {feedback[:200]}")

Menjalankan Lead Orchestrator (evaluasi laporan lengkap)...

[Orchestrator evaluasi] LULUS
  approval_status      : approved
  iteration            : 2
  feedback (200 char)  : 


---
## 8. Graph Pipeline Test

Uji build graph LangGraph dan jalankan **satu iterasi penuh** (satu putaran pipeline semua agent).

In [ ]:
from graphs.ebp_graph import build_graph, _route_from_orchestrator
print("Import graphs.ebp_graph: OK")

Import graphs.ebp_graph: OK


In [ ]:
# --- Test routing function secara unit ---

# Case 1: approved → END
route_approved = _route_from_orchestrator({"approval_status": "approved", "iteration": 1, "max_iterations": 3})
assert route_approved == "__end__", f"Expected __end__, got {route_approved}"
print("Route approved     :", route_approved)

# Case 2: max_iterations tercapai → END
route_maxed = _route_from_orchestrator({"approval_status": "rejected", "iteration": 3, "max_iterations": 3})
assert route_maxed == "__end__"
print("Route max_iter hit :", route_maxed)

# Case 3: rejected, masih ada iterasi → kembali ke market_scout
route_retry = _route_from_orchestrator({"approval_status": "rejected", "iteration": 1, "max_iterations": 3})
assert route_retry == "market_scout"
print("Route retry        :", route_retry)

print("\nSemua routing case: LULUS")

Route approved     : __end__
Route max_iter hit : __end__
Route retry        : market_scout

Semua routing case: LULUS


In [ ]:
# --- Build graph ---
graph = build_graph()
print("Graph berhasil dikompilasi:", type(graph).__name__)
print("Nodes:", list(graph.nodes.keys()) if hasattr(graph, 'nodes') else "(gunakan graph.get_graph() untuk detail)")

Graph berhasil dikompilasi: CompiledStateGraph
Nodes: ['__start__', 'lead_orchestrator', 'market_scout', 'strategic_architect', 'financial_analyst', 'ethics_agent']


---
## 9. End-to-End Integration Test

Jalankan graph secara penuh dengan **max_iterations=1** agar hanya 1 siklus pipeline yang berjalan. Ini mensimulasikan alur produksi nyata.

> **Perhatian:** Sel ini memanggil semua 5 agent dan internet search. Estimasi waktu: **5-15 menit**.

In [ ]:
import time

e2e_constraints = BussinessConstraints(
    sector_and_domain="EdTech / Platform Belajar Online",
    audience="Pelajar SMA dan mahasiswa di Indonesia (usia 16-22)",
    initial_prompt="Platform micro-learning berbasis video pendek untuk persiapan SNBT dan ujian sertifikasi profesional",
)

initial_state: EBPState = {
    "state_id": str(uuid.uuid4()),
    "user_id": "e2e_test",
    "bussiness_constraints": e2e_constraints,
    "market_scout_report": None,
    "strategic_report": None,
    "financial_analysis_report": None,
    "ethics_analysis_report": None,
    "approval_status": "pending",
    "orchestrator_feedback": None,
    "messages": [],
    "iteration": 0,
    "max_iterations": 1,  # 1 iterasi saja untuk test cepat
    "user_feedback": None,
}

print(f"State ID: {initial_state['state_id']}")
print("Menjalankan graph end-to-end (max_iterations=1)...")
print("=" * 60)

start = time.time()
final_state = graph.invoke(initial_state)
elapsed = time.time() - start

print(f"\nGraph selesai dalam {elapsed:.1f} detik")

State ID: aec01d95-ff13-4627-b0b9-35cbb8e62e0f
Menjalankan graph end-to-end (max_iterations=1)...


BadRequestError: Error code: 400 - {'error': {'message': "This model's maximum context length is 262144 tokens. However, you requested 81920 output tokens and your prompt contains at least 180225 input tokens, for a total of at least 262145 tokens. Please reduce the length of the input prompt or the number of requested output tokens. (parameter=input_tokens, value=180225)"}}

In [ ]:
# --- Validasi output end-to-end ---
print("=" * 60)
print("HASIL END-TO-END")
print("=" * 60)

print(f"approval_status     : {final_state.get('approval_status')}")
print(f"iteration           : {final_state.get('iteration')}")
print(f"messages count      : {len(final_state.get('messages', []))}")

ms = final_state.get("market_scout_report")
sa = final_state.get("strategic_report")
fa = final_state.get("financial_analysis_report")
ea = final_state.get("ethics_analysis_report")

print(f"\nmarket_scout_report      : {'Ada' if ms else 'TIDAK ADA'}")
print(f"strategic_report         : {'Ada' if sa else 'TIDAK ADA'}")
print(f"financial_analysis_report: {'Ada' if fa else 'TIDAK ADA'}")
print(f"ethics_analysis_report   : {'Ada' if ea else 'TIDAK ADA'}")

HASIL END-TO-END
approval_status     : rejected
iteration           : 1
messages count      : 30

market_scout_report      : Ada
strategic_report         : Ada
financial_analysis_report: Ada
ethics_analysis_report   : Ada


In [ ]:
# --- Tampilkan ringkasan tiap laporan ---
DIVIDER = "-" * 60

if ms:
    print(DIVIDER)
    print("MARKET SCOUT")
    print(DIVIDER)
    for i, idea in enumerate(ms.ideas, 1):
        print(f"  Ide {i}: {idea}")
    print(f"\n  Penjelasan: {ms.agent_explanation[:400]}")

if sa:
    print(DIVIDER)
    print("STRATEGIC ARCHITECT")
    print(DIVIDER)
    print(f"  SWOT  : {sa.swot_analysis[:400]}")
    print(f"  PESTEL: {sa.pastel_analysis[:400]}")

if fa:
    print(DIVIDER)
    print("FINANCIAL ANALYST")
    print(DIVIDER)
    print(f"  {fa.analysis_result[:500]}")

if ea:
    print(DIVIDER)
    print("ETHICS GUARDIAN")
    print(DIVIDER)
    print(f"  {ea.analysis_result[:500]}")

feedback = final_state.get("orchestrator_feedback") or ""
if feedback:
    print(DIVIDER)
    print("ORCHESTRATOR FEEDBACK")
    print(DIVIDER)
    print(f"  {feedback[:400]}")

------------------------------------------------------------
MARKET SCOUT
------------------------------------------------------------
  Ide 1: Platform Micro-Learning Video Pendek untuk Persiapan SNBT dengan Format TikTok-Style: Mengingat 77% penetrasi internet Indonesia dan preferensi Gen Z terhadap konten video pendek, platform ini menawarkan materi belajar SNBT dalam format 3-5 menit per topik dengan fitur gamifikasi, tryout adaptif, dan komunitas belajar. Pasar EdTech Indonesia diproyeksikan tumbuh dari USD 1,423.1 Juta (2025) menjadi USD 9,356.6 Juta (2034) dengan CAGR 23.28%, dengan segmen ujian masuk universitas menjadi katalis utama.
  Ide 2: Platform Sertifikasi Profesional Mikro untuk Mahasiswa dengan Partnership Industri: Menyasar demand sertifikasi profesional yang meningkat (3 juta+ sertifikasi telah diterbitkan di Indonesia), platform ini menyediakan kursus mikro terakreditasi untuk skill digital (data analytics, digital marketing, AI) dengan durasi 10-15 menit per modul

In [ ]:
# --- Assertions akhir end-to-end ---
assert final_state.get("approval_status") in ("approved", "rejected", "pending"), \
    "approval_status harus valid"
assert final_state.get("market_scout_report") is not None, \
    "Market scout report seharusnya terisi"
assert final_state.get("strategic_report") is not None, \
    "Strategic report seharusnya terisi"
assert final_state.get("financial_analysis_report") is not None, \
    "Financial report seharusnya terisi"
assert final_state.get("ethics_analysis_report") is not None, \
    "Ethics report seharusnya terisi"

print("SEMUA ASSERTION END-TO-END: LULUS")
print(f"\nSistem ClarioAI berjalan normal. Status akhir: {final_state['approval_status'].upper()}")

SEMUA ASSERTION END-TO-END: LULUS

Sistem ClarioAI berjalan normal. Status akhir: REJECTED
